# HealthConnect Week 8 - Final Data Science Model
**Takudzwa Jeffrey Gumbo | AnalystLab Africa Batch E | Data Science Track**

This notebook finalizes the tested Week 7 admission-model outcome. It does not claim clinical deployment readiness.

## 1. Week 7 to Week 8 transition
- Week 7 candidate testing showed near-chance discrimination.
- Cross-track validation with Data Analytics improved feature governance and reduced leakage risk.
- Week 8 selects the leakage-safer Gradient Boosting configuration as the **final documented candidate**, while retaining a **do-not-deploy** decision.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json, joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *

SEED=42
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SEARCH_ROOT=Path('/content/drive/MyDrive/Analyst Lab Africa')
except Exception:
    SEARCH_ROOT=Path('/mnt/data/week8_build/week6')

def find_unique(filename, root=SEARCH_ROOT):
    matches=list(root.rglob(filename))
    if len(matches)==0:
        raise FileNotFoundError(f'{filename} not found under {root}')
    if len(matches)>1:
        print('Multiple copies found:')
        for m in matches: print(m)
        raise RuntimeError(f'Resolve duplicate copies of {filename}')
    return matches[0]

data_path=find_unique('Integrated_Healthcare_Dataset.csv')
print('Using:',data_path)
df=pd.read_csv(data_path)
print('Shape:',df.shape)
print('Admissions:',df['Admitted'].value_counts().to_dict())

Mounted at /content/drive
Using: /content/drive/MyDrive/Analyst Lab Africa/HealthConnect Rwanda Week 6 (TJ Gumbo)/Data/Integrated/Integrated_Healthcare_Dataset.csv
Shape: (5000, 51)
Admissions: {0: 4414, 1: 586}


## 2. Final prediction point and feature decisions
Prediction point: **patient arrival / before initial clinical assessment**. Diagnosis and Severity are excluded because their timing may introduce leakage. Age is recalculated at appointment date; PriorVisits uses only earlier appointments.

In [2]:
df['Date']=pd.to_datetime(df['Date'])
df['DateOfBirth']=pd.to_datetime(df['DateOfBirth'])
df['AgeAtAppointment']=np.floor((df['Date']-df['DateOfBirth']).dt.days/365.2425).astype(int)
ordered=df.sort_values(['PatientID','Date','AppointmentID']).index
prior=pd.Series(index=df.index,dtype='float64')
prior.loc[ordered]=df.loc[ordered].groupby('PatientID').cumcount().values
df['PriorVisits']=prior.astype(int)
df['AppointmentMonth']=df['Date'].dt.month.astype(str)
df['AppointmentWeekday']=df['Date'].dt.day_name()
FEATURES=['AgeAtAppointment','Gender','HasChronicCondition','VisitType','FacilityID','DepartmentID','PriorVisits','AppointmentMonth','AppointmentWeekday']
NUM=['AgeAtAppointment','HasChronicCondition','PriorVisits']
CAT=[c for c in FEATURES if c not in NUM]
print(FEATURES)

['AgeAtAppointment', 'Gender', 'HasChronicCondition', 'VisitType', 'FacilityID', 'DepartmentID', 'PriorVisits', 'AppointmentMonth', 'AppointmentWeekday']


## 3. Final evaluation protocol
A stratified 80/20 split with random seed 42 is retained for comparability with the Week 7 cross-track re-test. Because the holdout partition had already been examined during Week 7, **5-fold cross-validation is treated as the stronger generalisation evidence**.

In [3]:
pre=ColumnTransformer([
 ('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),NUM),
 ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),CAT)
])
X=df[FEATURES]; y=df['Admitted'].astype(int)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED)
print('Holdout class counts:',yte.value_counts().sort_index().to_dict())
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)

Holdout class counts: {0: 883, 1: 117}


In [4]:
models={
 'Gradient Boosting':GradientBoostingClassifier(random_state=SEED),
 'Balanced Logistic Regression':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED)
}
rows=[]
for name,model in models.items():
    pipe=Pipeline([('pre',pre),('model',model)])
    pipe.fit(Xtr,ytr)
    prob=pipe.predict_proba(Xte)[:,1]
    pred=(prob>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(yte,pred).ravel()
    auc_cv=cross_val_score(pipe,X,y,cv=cv,scoring='roc_auc')
    pr_cv=cross_val_score(pipe,X,y,cv=cv,scoring='average_precision')
    rows.append({
      'Model':name,'Holdout_ROC_AUC':roc_auc_score(yte,prob),'Holdout_PR_AUC':average_precision_score(yte,prob),
      'Accuracy':accuracy_score(yte,pred),'Balanced_Accuracy':balanced_accuracy_score(yte,pred),
      'Precision':precision_score(yte,pred,zero_division=0),'Recall':recall_score(yte,pred,zero_division=0),'F1':f1_score(yte,pred,zero_division=0),
      'TN':tn,'FP':fp,'FN':fn,'TP':tp,'CV_ROC_AUC_Mean':auc_cv.mean(),'CV_ROC_AUC_Std':auc_cv.std(ddof=1),'CV_PR_AUC_Mean':pr_cv.mean()})
results=pd.DataFrame(rows)
display(results.round(4))

,Model,Holdout_ROC_AUC,Holdout_PR_AUC,Accuracy,Balanced_Accuracy,Precision,Recall,F1,TN,FP,FN,TP,CV_ROC_AUC_Mean,CV_ROC_AUC_Std,CV_PR_AUC_Mean
0,Gradient Boosting,0.5189,0.1199,0.882,0.4994,0.0000,0.0000,0.0000,882,1,117,0,0.5237,0.0246,0.1303
1,Balanced Logistic Regression,0.5024,0.1154,0.538,0.4974,0.1158,0.4444,0.1837,486,397,65,52,0.4986,0.0322,0.1245


## 4. Final selection and business suitability
The final documented candidate is Gradient Boosting because it has the best verified discrimination among the leakage-safer Week 8 configurations. However, ROC-AUC remains about **0.52** and the default-threshold model identifies **zero** positive admissions in the holdout set. The model is **not suitable for clinical or operational deployment**. Its value is methodological: it demonstrates a reproducible modelling workflow, leakage-aware feature governance, and evidence-based rejection of a weak predictive solution.

In [5]:
final=results.loc[results['Model']=='Gradient Boosting'].iloc[0]
assert 0.50 < final['Holdout_ROC_AUC'] < 0.54
assert 0.50 < final['CV_ROC_AUC_Mean'] < 0.55
print('Final documented candidate:',final['Model'])
print('Holdout ROC-AUC:',round(final['Holdout_ROC_AUC'],4))
print('5-fold CV ROC-AUC:',round(final['CV_ROC_AUC_Mean'],4),'+/-',round(final['CV_ROC_AUC_Std'],4))
print('Decision: NOT APPROVED FOR DEPLOYMENT')

Final documented candidate: Gradient Boosting
Holdout ROC-AUC: 0.5189
5-fold CV ROC-AUC: 0.5237 +/- 0.0246
Decision: NOT APPROVED FOR DEPLOYMENT


## 5. Cross-track integration
Data Analytics validation from Agu Precious directly influenced the final feature set: supported variables were retained, leakage-prone Diagnosis/Severity were held out, Age was aligned to appointment date, and time-safe PriorVisits/month/weekday were added. The resulting model remained weak, so the collaboration changed **feature governance and interpretation**, not the deployment decision.

## 6. Final limitations and recommendations
1. No attendance/no-show target exists in the supplied Rwanda appointment data, so the stated attendance use case cannot be modelled honestly from this source.
2. Admission discrimination remains close to chance.
3. The holdout set was reused during Week 7; cross-validation is therefore emphasized for generalisation.
4. Future work should obtain a true attendance target and stronger pre-admission predictors, freeze a fresh untouched test set, and only revisit deployment after materially stronger validation.

## 7. Final Week 8 conclusion
**Final outcome:** a professionally documented, leakage-safer admission-modelling candidate with reproducible evidence and a clear **do-not-deploy** decision. The project demonstrates that responsible Data Science includes rejecting weak models rather than forcing a positive result.